"""
Analysis 1: Macro Monthly Trend (full corpus, 2018-2026)
 
PURPOSE
-------
Establish the baseline temporal landscape of burnout discourse across the
WHOLE corpus, independent of any single named event. This answers:
  - "Are there seasonal or macro-level temporal trends across 2019-2026?"
  - Provides the monthly time series that Analysis 3 (cross-correlation)
    and Analysis 4 (event-synchronized averaging) will both consume.
 
This script does NOT need NVD/MITRE data -- it's purely derived from your
already-merged bat_posts_results_with_dates.csv.
 
OUTPUT
------
- monthly_trend.csv          : one row per year_month, with post counts,
                                burnout rate, and per-construct YES rates
- monthly_trend_summary.png  : a multi-panel chart (volume, overall burnout
                                rate, per-construct rates) -- saved as PNG
                                per project convention (individual PNGs)
- Printed console summary including:
    - months with the highest/lowest burnout rate
    - simple trend check (year-over-year mean comparison)
    - flags any months with suspiciously low post volume (coverage gaps),
      since a "spike" in rate from a thin month is not reliable signal
"""
 

A monthly_trend_summary.png with three stacked panels: post volume, overall burnout rate, and the four construct rates overlaid. And console output ranking the highest/lowest burnout-rate months, flagging any month with fewer than 50 posts as low-coverage (so a "spike" isn't actually just 8 posts in a sparse month), and a year-over-year mean table that's your first real answer to "are there seasonal/macro trends across 2019–2026."


In [2]:
!pip3 install matplotlib

Defaulting to user installation because normal site-packages is not writeable


In [3]:

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
 
# ============================================================
# CONFIG -- adjust if needed
# ============================================================
INPUT_CSV = "bat_posts_results_with_dates.csv"
COL_DATE = "created_utc"
CONSTRUCT_COLS = ["EX", "EMO", "COG", "MD"]
COL_BAT_SCORE = "bat_score"
BAT_HIGH_THRESHOLD = 2          # bat_score >= this counts as "burnout-positive"
MIN_POSTS_FOR_RELIABLE_MONTH = 50   # below this, flag the month as low-coverage
 
OUTPUT_CSV = "monthly_trend.csv"
OUTPUT_PNG = "monthly_trend_summary.png"
 

In [4]:

def _to_binary(val):
    if pd.isna(val):
        return 0
    if isinstance(val, (int, float)):
        return int(val != 0)
    s = str(val).strip().upper()
    return 1 if s in ("YES", "Y", "TRUE", "1") else 0
 

In [5]:

def load_data(path):
    df = pd.read_csv(path)
    print(f"Loaded {len(df):,} rows from {path}")
 
    raw = df[COL_DATE]
    if pd.api.types.is_numeric_dtype(raw):
        df[COL_DATE] = pd.to_datetime(raw, unit="s", errors="coerce", utc=True)
    else:
        numeric_like = pd.to_numeric(raw, errors="coerce")
        if numeric_like.notna().mean() > 0.9:
            df[COL_DATE] = pd.to_datetime(numeric_like, unit="s", errors="coerce", utc=True)
        else:
            df[COL_DATE] = pd.to_datetime(raw, errors="coerce", utc=True)
 
    n_bad = df[COL_DATE].isna().sum()
    if n_bad > 0:
        print(f"WARNING: dropping {n_bad:,} rows with unparseable dates "
              f"({n_bad/len(df)*100:.1f}%).")
    df = df.dropna(subset=[COL_DATE])
    df[COL_DATE] = df[COL_DATE].dt.tz_localize(None)
 
    for col in CONSTRUCT_COLS:
        if col in df.columns:
            df[col] = df[col].apply(_to_binary)
        else:
            print(f"WARNING: construct column '{col}' not found.")
 
    df["year_month"] = df[COL_DATE].dt.to_period("M")
    print(f"Date range: {df[COL_DATE].min().date()} to {df[COL_DATE].max().date()}")
    print(f"Usable rows: {len(df):,}\n")
    return df

In [6]:
def build_monthly_trend(df):
    rows = []
    for ym, group in df.groupby("year_month"):
        n_total = len(group)
        row = {
            "year_month": str(ym),
            "n_posts": n_total,
        }
        for col in CONSTRUCT_COLS:
            if col in df.columns:
                n_yes = int(group[col].sum())
                row[f"{col}_yes_n"] = n_yes
                row[f"{col}_yes_rate"] = round(n_yes / n_total, 4) if n_total else np.nan
 
        if COL_BAT_SCORE in df.columns:
            n_high = int((group[COL_BAT_SCORE] >= BAT_HIGH_THRESHOLD).sum())
            row["bat_score_mean"] = round(group[COL_BAT_SCORE].mean(), 4)
            row["burnout_rate"] = round(n_high / n_total, 4) if n_total else np.nan
            row["burnout_high_n"] = n_high
 
        row["low_coverage_flag"] = n_total < MIN_POSTS_FOR_RELIABLE_MONTH
        rows.append(row)
 
    trend_df = pd.DataFrame(rows).sort_values("year_month").reset_index(drop=True)
    return trend_df
 

In [7]:

def print_summary(trend_df):
    print("=" * 70)
    print("MONTHLY TREND SUMMARY")
    print("=" * 70)
    print(trend_df.to_string(index=False))
 
    n_low_cov = trend_df["low_coverage_flag"].sum()
    print(f"\n{n_low_cov} of {len(trend_df)} months have fewer than "
          f"{MIN_POSTS_FOR_RELIABLE_MONTH} posts (low coverage -- treat "
          "their rates cautiously, they're noisy).")
    if n_low_cov > 0:
        print("Low-coverage months:")
        print(trend_df.loc[trend_df["low_coverage_flag"],
                            ["year_month", "n_posts"]].to_string(index=False))
 
    # Reliable-only subset for highest/lowest ranking
    reliable = trend_df.loc[~trend_df["low_coverage_flag"]].copy()
 
    if "burnout_rate" in reliable.columns and len(reliable) > 0:
        print("\n--- Top 5 months by overall burnout_rate (reliable months only) ---")
        print(reliable.nlargest(5, "burnout_rate")[
            ["year_month", "n_posts", "burnout_rate"]].to_string(index=False))
 
        print("\n--- Bottom 5 months by overall burnout_rate (reliable months only) ---")
        print(reliable.nsmallest(5, "burnout_rate")[
            ["year_month", "n_posts", "burnout_rate"]].to_string(index=False))
 
    # Year-over-year mean comparison (simple secular trend check)
    print("\n--- Year-over-year mean rates (reliable months only) ---")
    reliable["year"] = reliable["year_month"].str[:4]
    yearly_cols = ["burnout_rate"] + [f"{c}_yes_rate" for c in CONSTRUCT_COLS
                                       if f"{c}_yes_rate" in reliable.columns]
    yearly = reliable.groupby("year")[yearly_cols].mean().round(4)
    print(yearly.to_string())

In [8]:

def plot_trend(trend_df, out_path):
    fig, axes = plt.subplots(3, 1, figsize=(13, 11), sharex=True)
    x = trend_df["year_month"]
 
    # Panel 1: post volume
    axes[0].bar(x, trend_df["n_posts"], color="#6b7280")
    axes[0].set_ylabel("Posts per month")
    axes[0].set_title("Monthly post volume (full corpus)")
 
    # Panel 2: overall burnout rate
    if "burnout_rate" in trend_df.columns:
        axes[1].plot(x, trend_df["burnout_rate"], color="#dc2626", marker="o", markersize=3)
        axes[1].set_ylabel(f"Burnout rate\n(bat_score >= {BAT_HIGH_THRESHOLD})")
        axes[1].set_title("Monthly overall burnout rate")
 
    # Panel 3: per-construct rates
    colors = {"EX": "#2563eb", "EMO": "#dc2626", "COG": "#16a34a", "MD": "#9333ea"}
    for col in CONSTRUCT_COLS:
        rate_col = f"{col}_yes_rate"
        if rate_col in trend_df.columns:
            axes[2].plot(x, trend_df[rate_col], label=col, color=colors.get(col), linewidth=1.5)
    axes[2].set_ylabel("Construct YES rate")
    axes[2].set_title("Monthly per-construct rates (EX, EMO, COG, MD)")
    axes[2].legend(loc="upper left")
 
    # Thin out x tick labels for readability
    n = len(x)
    step = max(1, n // 24)
    for ax in axes:
        ax.set_xticks(range(0, n, step))
        ax.set_xticklabels(x[::step], rotation=45, ha="right", fontsize=8)
        ax.grid(alpha=0.3)
 
    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    print(f"\nSaved chart -> {out_path}")
 
 
if __name__ == "__main__":
    df = load_data(INPUT_CSV)
    trend_df = build_monthly_trend(df)
    print_summary(trend_df)
    trend_df.to_csv(OUTPUT_CSV, index=False)
    print(f"\nSaved data -> {OUTPUT_CSV}")
    plot_trend(trend_df, OUTPUT_PNG)

Loaded 135,897 rows from bat_posts_results_with_dates.csv
Date range: 2018-01-01 to 2026-04-25
Usable rows: 135,897

MONTHLY TREND SUMMARY
year_month  n_posts  EX_yes_n  EX_yes_rate  EMO_yes_n  EMO_yes_rate  COG_yes_n  COG_yes_rate  MD_yes_n  MD_yes_rate  bat_score_mean  burnout_rate  burnout_high_n  low_coverage_flag
   2018-01     1011        52       0.0514        141        0.1395         23        0.0227        40       0.0396          0.2532        0.0584              59              False
   2018-02      994        47       0.0473        140        0.1408         22        0.0221        33       0.0332          0.2435        0.0503              50              False
   2018-03     1193        68       0.0570        165        0.1383         32        0.0268        48       0.0402          0.2624        0.0612              73              False
   2018-04     1139        59       0.0518        173        0.1519         26        0.0228        34       0.0299          0.2564      

In [13]:
import sys
!{sys.executable} -m pip install scipy statsmodels ruptures

Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.4/39.4 MB 25.6 MB/s  0:00:01 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 27.7 MB/s  0:00:00eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [statsmodels] [statsmodels]


The headline finding: a real secular trend, not just noise
Burnout rate climbs steadily and substantially from 2018 (6.4%) to 2025 (8.8%) — a roughly 38% relative increase over seven years, then ticks down slightly into 2026 (8.2%, though that's a partial year, only through April). This isn't a flat baseline with occasional spikes; it's a genuine upward drift across most of your corpus's lifespan. That's a real, reportable finding independent of any single CVE event, and it directly answers your "are there seasonal or macro-level trends" research question with a clear yes.
Two things make this more interesting than a generic "burnout is rising" claim, and worth digging into before you write it up:
Post volume is also rising over the same period (1,112/month in 2018 → ~1,690/month by 2024-2026). Since burnout_rate is already a rate (not a raw count), this isn't a denominator artifact — but it does raise the question of whether subreddit growth itself (more total cybersecurity professionals on Reddit, or specific subreddits like r/cybersecurity growing in popularity) is partially driving the burnout rate trend, versus something specific to working conditions. Worth checking: does the trend hold if you look subreddit by subreddit, or is it driven by one subreddit's growth diluting/concentrating the mix?
The four constructs aren't moving in lockstep. Looking at the year-over-year table: EX rises from 5.9% (2018) to a 2023 peak of 7.3%, MD nearly doubles (4.0% → 6.3%), while EMO is comparatively flat/slightly declining in its later years (15.6% in 2018 → 16.9% in 2022 → drifting back to 16.4% in 2026) and COG creeps up only modestly (2.6% → 3.3%). That's exactly the kind of "the type of burnout is shifting, not just the amount" story your advisor wants — MD (mental distance/cynicism) and EX (exhaustion) are growing proportionally faster than EMO (emotional impairment), suggesting a possible shift toward disengagement/withdrawal-flavored burnout rather than acute emotional distress. That's worth a sentence or two in your writeup as a genuinely novel observation, separate from the CVE-event analysis entirely.
A methodological note on the chart itself
Looking at the bottom panel, EMO sits an order of magnitude above the other three lines (consistently 14-20% vs. 2-9% for the others) — visually this compresses COG and MD into a cramped band at the bottom and makes it hard to see their individual shape changes over time. For your actual thesis figure, I'd suggest either a log y-axis, or four separate small-multiple panels (one per construct, own y-axis), or a normalized version (each construct's rate divided by its own 2018 baseline, so you're comparing % change rather than absolute level). Happy to build that as a follow-up chart if useful — it's a presentation fix, not a re-analysis.

In [ ]:
# #find the how much burnout indicative posts increased or decreased 2018-2026? or when it increased and then decreased?
# #RQ1.2: Do the four BAT constructs (EX, EMO, COG, MD) grow at different rates over this period, indicating a shift in the compositional makeup of burnout rather than a uniform increase? 
# # find the numbers


# """
# burnout_trend_analysis.py
# ==========================

# Answers three questions from monthly-aggregated BAT construct rates
# (EX, EMO, COG, MD), burnout_rate, and bat_score_mean, 2018-01 to 2026-04:

#   Q1. Has the prevalence of burnout-indicative discourse increased
#       significantly between 2018 and 2026?
#   Q2. Do the four BAT constructs (EX, EMO, COG, MD) grow at different
#       rates, indicating a compositional shift rather than a uniform
#       increase?
#   Q3. Is that compositional shift gradual/secular, or punctuated
#       (concentrated around specific years / CVE events)?

# INPUT
# -----
# A CSV with one row per calendar month and (at minimum) the columns:
#     year_month, n_posts,
#     EX_yes_rate, EMO_yes_rate, COG_yes_rate, MD_yes_rate,
#     bat_score_mean, burnout_rate

# METHODS (documented so they can be cited directly in the thesis)
# ------------------------------------------------------------------
# Q1  - OLS trend line with Newey-West (HAC) standard errors, to get a
#       slope estimate that is valid under autocorrelated monthly errors.
#     - Classic Mann-Kendall test (nonparametric, monotonic-trend test;
#       tie-corrected) as a distribution-free cross-check.
#     - Theil-Sen estimator for a trend slope that is robust to outliers.

# Q2  - Each construct's own HAC-OLS / Mann-Kendall / Theil-Sen trend
#       (same machinery as Q1), to get a per-construct annualized growth
#       rate.
#     - A single pooled regression (long format) of
#           rate ~ time_years * construct
#       with cluster-robust SEs (clustered by month, since the four
#       construct series in a given month share the same underlying
#       corpus and are not independent draws). A joint Wald test on the
#       time_years:construct interaction terms formally tests "do the
#       four constructs have the same slope?" -- rejecting H0 is the
#       statistical signature of a compositional shift.
#     - Construct "shares" (rate_X / sum of all four rates each month)
#       and a disengagement index (EX+MD)/(EMO+COG), each given the same
#       HAC-OLS/MK trend test, to see which constructs are gaining vs.
#       losing relative weight.

# Q3  - Quadratic-term test: rate ~ time_years + time_years^2 (HAC-OLS).
#       A significant positive quadratic term indicates the trend is
#       accelerating (consistent with "punctuated/intensifying") rather
#       than linear/secular.
#     - Single unknown-breakpoint structural break test (Quandt-Andrews
#       style): grid search over all candidate split months for the
#       breakpoint that maximizes the SSE improvement of a two-segment
#       (separate intercept+slope each side) piecewise-linear fit over a
#       single trend line; significance assessed via a moving-block
#       bootstrap under the null of one constant linear trend (preserves
#       autocorrelation structure when generating the null distribution).
#     - The detected breakpoint date is cross-referenced against four
#       named CVE events (Log4Shell, SolarWinds, MOVEit, CrowdStrike) to
#       see whether structural change aligns with those events.
#     - A descriptive (non-bootstrapped) BIC comparison across 0/1/2
#       breakpoints and a rolling 24-month slope are reported as
#       supporting/exploratory evidence.

# Outputs land in OUT_DIR: summary tables (CSV), a text report, and PNG
# figures.

# Usage
# -----
#     python3 burnout_trend_analysis.py [path_to_monthly_trend.csv] [out_dir]
# """

# import sys
# import warnings
# from pathlib import Path

# import numpy as np
# import pandas as pd
# import matplotlib
# matplotlib.use("Agg")
# import matplotlib.pyplot as plt
# import matplotlib.dates as mdates
# from scipy import stats
# import statsmodels.api as sm
# import statsmodels.formula.api as smf

# warnings.filterwarnings("ignore")

# # ----------------------------------------------------------------------
# # Config
# # ----------------------------------------------------------------------

# CONSTRUCTS = ["EX", "EMO", "COG", "MD"]
# RATE_COLS = {c: f"{c}_yes_rate" for c in CONSTRUCTS}

# CONSTRUCT_LABELS = {
#     "EX": "Exhaustion",
#     "EMO": "Emotional Impairment",
#     "COG": "Cognitive Impairment",
#     "MD": "Mental Distance",
# }

# KNOWN_EVENTS = {
#     "SolarWinds": "2020-12",
#     "Log4Shell": "2021-12",
#     "MOVEit": "2023-05",
#     "CrowdStrike": "2024-07",
# }

# HAC_MAXLAGS = 12        # one year of monthly autocorrelation
# N_BOOT = 400             # bootstrap replicates for breakpoint significance
# BLOCK_SIZE = 8           # moving-block bootstrap block length (months)
# MIN_SEGMENT = 9          # minimum months on each side of a candidate break
# RNG_SEED = 20260618

# rng = np.random.default_rng(RNG_SEED)


# # ----------------------------------------------------------------------
# # Data loading
# # ----------------------------------------------------------------------

# def load_data(path):
#     df = pd.read_csv(path)
#     df["date"] = pd.to_datetime(df["year_month"], format="%Y-%m")
#     df = df.sort_values("date").reset_index(drop=True)
#     df["t_month"] = np.arange(len(df))
#     df["t_year"] = df["t_month"] / 12.0
#     df["year"] = df["date"].dt.year
#     return df


# # ----------------------------------------------------------------------
# # Q1 / generic trend-testing primitives
# # ----------------------------------------------------------------------

# def hac_ols_trend(y, x_years):
#     """OLS y ~ x_years with Newey-West (HAC) SEs. Returns dict of stats."""
#     X = sm.add_constant(np.asarray(x_years, dtype=float))
#     model = sm.OLS(np.asarray(y, dtype=float), X)
#     res = model.fit(cov_type="HAC", cov_kwds={"maxlags": HAC_MAXLAGS})
#     slope = res.params[1]
#     se = res.bse[1]
#     pval = res.pvalues[1]
#     ci_lo, ci_hi = res.conf_int()[1]
#     return {
#         "slope_per_year": slope,
#         "se": se,
#         "p_value": pval,
#         "ci_lo": ci_lo,
#         "ci_hi": ci_hi,
#         "r_squared": res.rsquared,
#         "n": len(y),
#     }


# def mann_kendall(y):
#     """Classic tie-corrected Mann-Kendall monotonic trend test."""
#     y = np.asarray(y, dtype=float)
#     n = len(y)
#     s = 0
#     for i in range(n - 1):
#         s += np.sum(np.sign(y[i + 1:] - y[i]))

#     _, counts = np.unique(y, return_counts=True)
#     tie_term = np.sum(counts * (counts - 1) * (2 * counts + 5))
#     var_s = (n * (n - 1) * (2 * n + 5) - tie_term) / 18.0

#     if s > 0:
#         z = (s - 1) / np.sqrt(var_s)
#     elif s < 0:
#         z = (s + 1) / np.sqrt(var_s)
#     else:
#         z = 0.0

#     p = 2 * (1 - stats.norm.cdf(abs(z)))
#     tau = s / (0.5 * n * (n - 1))
#     return {"S": s, "z": z, "p_value": p, "tau": tau}


# def theil_sen(y, x_years):
#     slope, intercept, lo, hi = stats.theilslopes(np.asarray(y, dtype=float),
#                                                   np.asarray(x_years, dtype=float))
#     return {"slope_per_year": slope, "intercept": intercept, "ci_lo": lo, "ci_hi": hi}


# def full_trend_report(y, x_years, label):
#     hac = hac_ols_trend(y, x_years)
#     mk = mann_kendall(y)
#     ts = theil_sen(y, x_years)
#     return {
#         "series": label,
#         "hac_slope_per_year": hac["slope_per_year"],
#         "hac_p_value": hac["p_value"],
#         "hac_ci_lo": hac["ci_lo"],
#         "hac_ci_hi": hac["ci_hi"],
#         "r_squared": hac["r_squared"],
#         "mk_tau": mk["tau"],
#         "mk_p_value": mk["p_value"],
#         "theilsen_slope_per_year": ts["slope_per_year"],
#     }


# def quadratic_acceleration_test(y, x_years, label):
#     X = np.column_stack([x_years, x_years ** 2])
#     X = sm.add_constant(X)
#     res = sm.OLS(np.asarray(y, dtype=float), X).fit(
#         cov_type="HAC", cov_kwds={"maxlags": HAC_MAXLAGS}
#     )
#     return {
#         "series": label,
#         "linear_coef": res.params[1],
#         "quadratic_coef": res.params[2],
#         "quadratic_p_value": res.pvalues[2],
#         "r_squared": res.rsquared,
#     }


# # ----------------------------------------------------------------------
# # Q3 / structural break primitives
# # ----------------------------------------------------------------------

# def _segment_sse(x, y):
#     """SSE of a single OLS line y ~ x (with intercept)."""
#     if len(x) < 2:
#         return 0.0
#     slope, intercept = np.polyfit(x, y, 1)
#     resid = y - (slope * x + intercept)
#     return float(np.sum(resid ** 2)), slope, intercept


# def single_line_fit(x, y):
#     sse, slope, intercept = _segment_sse(x, y)
#     return sse, slope, intercept


# def best_single_breakpoint(x, y, min_seg=MIN_SEGMENT):
#     """Grid search the breakpoint index that minimizes total two-segment SSE.
#     Two segments each get their own intercept+slope (discontinuous fit)."""
#     n = len(x)
#     best = None
#     for bp in range(min_seg, n - min_seg):
#         sse_left, slope_l, int_l = _segment_sse(x[:bp], y[:bp])
#         sse_right, slope_r, int_r = _segment_sse(x[bp:], y[bp:])
#         total = sse_left + sse_right
#         if best is None or total < best["sse"]:
#             best = {
#                 "bp_index": bp,
#                 "sse": total,
#                 "slope_left": slope_l,
#                 "slope_right": slope_r,
#                 "intercept_left": int_l,
#                 "intercept_right": int_r,
#             }
#     return best


# def bic_for_k_breaks(x, y, k):
#     """Descriptive BIC for the best-fitting piecewise-linear model with
#     exactly k breakpoints (k in {0,1,2}), each segment free intercept+slope.
#     Exhaustive search for k<=2, with a minimum segment length."""
#     n = len(x)
#     min_seg = MIN_SEGMENT

#     if k == 0:
#         sse, _, _ = _segment_sse(x, y)
#         bps = []
#     elif k == 1:
#         best = best_single_breakpoint(x, y, min_seg)
#         sse = best["sse"]
#         bps = [best["bp_index"]]
#     elif k == 2:
#         best_sse = None
#         best_bps = None
#         candidates = range(min_seg, n - min_seg)
#         for i, bp1 in enumerate(candidates):
#             for bp2 in range(bp1 + min_seg, n - min_seg):
#                 sse1, _, _ = _segment_sse(x[:bp1], y[:bp1])
#                 sse2, _, _ = _segment_sse(x[bp1:bp2], y[bp1:bp2])
#                 sse3, _, _ = _segment_sse(x[bp2:], y[bp2:])
#                 total = sse1 + sse2 + sse3
#                 if best_sse is None or total < best_sse:
#                     best_sse = total
#                     best_bps = [bp1, bp2]
#         sse = best_sse
#         bps = best_bps
#     else:
#         raise ValueError("k must be 0, 1, or 2")

#     k_params = 2 * (k + 1)  # intercept+slope per segment
#     bic = n * np.log(sse / n) + k_params * np.log(n)
#     return {"k": k, "sse": sse, "bic": bic, "breakpoints_idx": bps}


# def moving_block_bootstrap_resample(residuals, block_size, n, rng):
#     n_blocks = int(np.ceil(n / block_size))
#     max_start = len(residuals) - block_size
#     starts = rng.integers(0, max_start + 1, size=n_blocks)
#     pieces = [residuals[s:s + block_size] for s in starts]
#     return np.concatenate(pieces)[:n]


# def breakpoint_significance_test(x, y, n_boot=N_BOOT, block_size=BLOCK_SIZE,
#                                    min_seg=MIN_SEGMENT, rng=rng):
#     """Quandt-Andrews-style test for one unknown breakpoint, with a
#     moving-block bootstrap null distribution (preserves autocorrelation)."""
#     n = len(x)

#     sse_null, slope_null, intercept_null = single_line_fit(x, y)
#     best = best_single_breakpoint(x, y, min_seg)
#     observed_improvement = sse_null - best["sse"]

#     fitted_null = slope_null * x + intercept_null
#     resid_null = y - fitted_null

#     boot_improvements = np.empty(n_boot)
#     for b in range(n_boot):
#         boot_resid = moving_block_bootstrap_resample(resid_null, block_size, n, rng)
#         y_boot = fitted_null + boot_resid
#         sse_null_b, _, _ = single_line_fit(x, y_boot)
#         best_b = best_single_breakpoint(x, y_boot, min_seg)
#         boot_improvements[b] = sse_null_b - best_b["sse"]

#     p_value = float(np.mean(boot_improvements >= observed_improvement))

#     return {
#         "bp_index": best["bp_index"],
#         "slope_before": best["slope_left"],
#         "slope_after": best["slope_right"],
#         "sse_null": sse_null,
#         "sse_break": best["sse"],
#         "observed_sse_improvement": observed_improvement,
#         "bootstrap_p_value": p_value,
#         "n_boot": n_boot,
#     }


# def rolling_slope(dates, y, window=24):
#     n = len(y)
#     out = np.full(n, np.nan)
#     x_years_full = np.arange(n) / 12.0
#     for i in range(window, n + 1):
#         xw = x_years_full[i - window:i]
#         yw = y[i - window:i]
#         slope, _ = np.polyfit(xw, yw, 1)
#         out[i - 1] = slope
#     return out


# def nearest_event(date, events=KNOWN_EVENTS, tol_months=3):
#     best_name, best_dist = None, None
#     for name, ym in events.items():
#         ev_date = pd.to_datetime(ym, format="%Y-%m")
#         dist = abs((date.year - ev_date.year) * 12 + (date.month - ev_date.month))
#         if best_dist is None or dist < best_dist:
#             best_dist, best_name = dist, name
#     within_tol = best_dist is not None and best_dist <= tol_months
#     return best_name, best_dist, within_tol


# # ----------------------------------------------------------------------
# # Q1: overall prevalence trend
# # ----------------------------------------------------------------------

# def run_q1(df, out_dir):
#     rows = []
#     for col, label in [("burnout_rate", "burnout_rate"), ("bat_score_mean", "bat_score_mean")]:
#         rows.append(full_trend_report(df[col].values, df["t_year"].values, label))
#     q1_table = pd.DataFrame(rows)
#     q1_table.to_csv(out_dir / "q1_overall_trend.csv", index=False)

#     # year-over-year averages (note final year may be partial)
#     yoy = df.groupby("year")[["burnout_rate", "bat_score_mean", "n_posts"]].mean()
#     yoy["n_months"] = df.groupby("year")["t_month"].count()
#     yoy.to_csv(out_dir / "q1_year_over_year_means.csv")

#     first_year_rate = df[df["year"] == df["year"].min()]["burnout_rate"].mean()
#     last_full_year = df[df["year"] < df["year"].max()]["year"].max()
#     last_full_year_rate = df[df["year"] == last_full_year]["burnout_rate"].mean()
#     pct_change = 100 * (last_full_year_rate - first_year_rate) / first_year_rate

#     return {
#         "table": q1_table,
#         "yoy": yoy,
#         "first_year": int(df["year"].min()),
#         "last_full_year": int(last_full_year),
#         "first_year_rate": first_year_rate,
#         "last_full_year_rate": last_full_year_rate,
#         "pct_change": pct_change,
#     }


# # ----------------------------------------------------------------------
# # Q2: differential construct growth / compositional shift
# # ----------------------------------------------------------------------

# def run_q2(df, out_dir):
#     # per-construct trend
#     rows = []
#     for c in CONSTRUCTS:
#         rows.append(full_trend_report(df[RATE_COLS[c]].values, df["t_year"].values,
#                                        CONSTRUCT_LABELS[c]))
#     construct_trends = pd.DataFrame(rows)
#     construct_trends.to_csv(out_dir / "q2_construct_trends.csv", index=False)

#     # pooled long-format regression with interaction, cluster-robust by month
#     long_rows = []
#     for c in CONSTRUCTS:
#         for _, r in df.iterrows():
#             long_rows.append({
#                 "month_id": r["t_month"],
#                 "t_year": r["t_year"],
#                 "construct": c,
#                 "rate": r[RATE_COLS[c]],
#             })
#     long_df = pd.DataFrame(long_rows)
#     long_df["construct"] = pd.Categorical(long_df["construct"],
#                                            categories=CONSTRUCTS, ordered=False)

#     model = smf.ols("rate ~ t_year + C(construct) + t_year:C(construct)", data=long_df)
#     res = model.fit(cov_type="cluster", cov_kwds={"groups": long_df["month_id"]})
#     wald = res.wald_test_terms(skip_single=False)
#     interaction_row = [r for r in wald.table.index if "t_year:C(construct)" in r]
#     interaction_pvalue = float(wald.table.loc[interaction_row[0], "pvalue"]) if interaction_row else np.nan

#     with open(out_dir / "q2_pooled_regression_summary.txt", "w") as f:
#         f.write(str(res.summary()))
#         f.write("\n\nJoint Wald test (term-level):\n")
#         f.write(str(wald.table))
#         f.write(f"\n\nInteraction (slope-difference) joint p-value: {interaction_pvalue:.6f}\n")

#     # construct shares of total signal
#     total = df[[RATE_COLS[c] for c in CONSTRUCTS]].sum(axis=1)
#     shares = pd.DataFrame({c: df[RATE_COLS[c]] / total for c in CONSTRUCTS})
#     shares["t_year"] = df["t_year"]
#     shares["date"] = df["date"]

#     share_rows = []
#     for c in CONSTRUCTS:
#         share_rows.append(full_trend_report(shares[c].values, shares["t_year"].values,
#                                              f"{c}_share"))
#     share_trends = pd.DataFrame(share_rows)
#     share_trends.to_csv(out_dir / "q2_share_trends.csv", index=False)

#     # disengagement index = (EX+MD) / (EMO+COG), using rates
#     disengagement = (df[RATE_COLS["EX"]] + df[RATE_COLS["MD"]]) / \
#                     (df[RATE_COLS["EMO"]] + df[RATE_COLS["COG"]])
#     di_trend = full_trend_report(disengagement.values, df["t_year"].values,
#                                   "disengagement_index_(EX+MD)/(EMO+COG)")
#     pd.DataFrame([di_trend]).to_csv(out_dir / "q2_disengagement_index_trend.csv", index=False)

#     # index each construct rate to its own first-year baseline = 100
#     base_year = df["year"].min()
#     baseline = df[df["year"] == base_year][[RATE_COLS[c] for c in CONSTRUCTS]].mean()
#     indexed = pd.DataFrame({c: 100 * df[RATE_COLS[c]] / baseline[RATE_COLS[c]] for c in CONSTRUCTS})
#     indexed["date"] = df["date"]
#     indexed.to_csv(out_dir / "q2_indexed_construct_rates.csv", index=False)

#     return {
#         "construct_trends": construct_trends,
#         "interaction_pvalue": interaction_pvalue,
#         "wald_table": wald.table,
#         "share_trends": share_trends,
#         "disengagement_series": disengagement,
#         "disengagement_trend": di_trend,
#         "indexed": indexed,
#         "shares": shares,
#     }


# # ----------------------------------------------------------------------
# # Q3: gradual vs. punctuated
# # ----------------------------------------------------------------------

# def run_q3(df, disengagement_series, out_dir):
#     x = df["t_year"].values

#     targets = {
#         "burnout_rate": df["burnout_rate"].values,
#         "bat_score_mean": df["bat_score_mean"].values,
#         "disengagement_index": disengagement_series.values,
#     }

#     accel_rows = []
#     bic_rows = []
#     bp_results = {}
#     for name, y in targets.items():
#         accel_rows.append(quadratic_acceleration_test(y, x, name))

#         for k in (0, 1, 2):
#             b = bic_for_k_breaks(x, y, k)
#             bic_rows.append({
#                 "series": name, "k_breaks": k, "sse": b["sse"], "bic": b["bic"],
#                 "breakpoint_dates": [df["date"].iloc[i].strftime("%Y-%m") for i in b["breakpoints_idx"]],
#             })

#         bp_test = breakpoint_significance_test(x, y)
#         bp_date = df["date"].iloc[bp_test["bp_index"]]
#         ev_name, ev_dist, within_tol = nearest_event(bp_date)
#         bp_results[name] = {
#             **bp_test,
#             "breakpoint_date": bp_date.strftime("%Y-%m"),
#             "nearest_event": ev_name,
#             "months_from_nearest_event": ev_dist,
#             "within_3_months_of_event": within_tol,
#         }

#     accel_table = pd.DataFrame(accel_rows)
#     accel_table.to_csv(out_dir / "q3_acceleration_test.csv", index=False)

#     bic_table = pd.DataFrame(bic_rows)
#     bic_table.to_csv(out_dir / "q3_bic_breakpoint_comparison.csv", index=False)

#     bp_table = pd.DataFrame(bp_results).T
#     bp_table.to_csv(out_dir / "q3_breakpoint_significance.csv")

#     # rolling slope for visualization / supporting evidence
#     roll = rolling_slope(df["date"].values, df["burnout_rate"].values, window=24)
#     roll_di = rolling_slope(df["date"].values, disengagement_series.values, window=24)

#     return {
#         "acceleration": accel_table,
#         "bic": bic_table,
#         "breakpoints": bp_table,
#         "rolling_slope_burnout": roll,
#         "rolling_slope_disengagement": roll_di,
#     }


# # ----------------------------------------------------------------------
# # Plots
# # ----------------------------------------------------------------------

# def _mark_events(ax):
#     for name, ym in KNOWN_EVENTS.items():
#         d = pd.to_datetime(ym, format="%Y-%m")
#         ax.axvline(d, color="gray", linestyle="--", linewidth=0.8, alpha=0.7)
#         ax.text(d, ax.get_ylim()[1], name, rotation=90, fontsize=7,
#                 va="top", ha="right", color="gray")


# def plot_overall_trend(df, out_dir):
#     fig, ax1 = plt.subplots(figsize=(11, 5))
#     ax1.plot(df["date"], df["burnout_rate"], color="firebrick", label="burnout_rate")
#     slope, intercept = np.polyfit(df["t_year"], df["burnout_rate"], 1)
#     ax1.plot(df["date"], slope * df["t_year"] + intercept, color="firebrick",
#               linestyle=":", linewidth=1.5, label="linear trend")
#     ax1.set_ylabel("burnout_rate", color="firebrick")
#     ax1.tick_params(axis="y", labelcolor="firebrick")

#     ax2 = ax1.twinx()
#     ax2.plot(df["date"], df["bat_score_mean"], color="steelblue", label="bat_score_mean")
#     ax2.set_ylabel("bat_score_mean", color="steelblue")
#     ax2.tick_params(axis="y", labelcolor="steelblue")

#     _mark_events(ax1)
#     ax1.set_title("Q1: Overall burnout-discourse prevalence, 2018-2026")
#     fig.legend(loc="upper left", bbox_to_anchor=(0.08, 0.95))
#     fig.tight_layout()
#     fig.savefig(out_dir / "fig1_overall_trend.png", dpi=150)
#     plt.close(fig)


# def plot_indexed_constructs(indexed, out_dir):
#     fig, ax = plt.subplots(figsize=(11, 5))
#     for c in CONSTRUCTS:
#         ax.plot(indexed["date"], indexed[c], label=f"{c} ({CONSTRUCT_LABELS[c]})")
#     ax.axhline(100, color="black", linewidth=0.6, linestyle="--")
#     _mark_events(ax)
#     ax.set_ylabel("Index (first-year monthly average = 100)")
#     ax.set_title("Q2: Indexed growth of the four BAT constructs")
#     ax.legend()
#     fig.tight_layout()
#     fig.savefig(out_dir / "fig2_indexed_constructs.png", dpi=150)
#     plt.close(fig)


# def plot_shares(shares, out_dir):
#     fig, ax = plt.subplots(figsize=(11, 5))
#     for c in CONSTRUCTS:
#         ax.plot(shares["date"], shares[c], label=f"{c} share")
#     _mark_events(ax)
#     ax.set_ylabel("Share of total BAT signal")
#     ax.set_title("Q2: Relative share of each construct over time")
#     ax.legend()
#     fig.tight_layout()
#     fig.savefig(out_dir / "fig3_construct_shares.png", dpi=150)
#     plt.close(fig)


# def plot_disengagement_and_breaks(df, disengagement_series, bp_table, out_dir):
#     fig, ax = plt.subplots(figsize=(11, 5))
#     ax.plot(df["date"], disengagement_series, color="darkorange", label="(EX+MD)/(EMO+COG)")
#     bp_idx = int(bp_table.loc["disengagement_index", "bp_index"])
#     bp_date = df["date"].iloc[bp_idx]
#     ax.axvline(bp_date, color="darkorange", linewidth=2, label=f"detected break: {bp_date:%Y-%m}")
#     _mark_events(ax)
#     ax.set_ylabel("Disengagement index")
#     ax.set_title("Q3: Disengagement index with detected structural break")
#     ax.legend()
#     fig.tight_layout()
#     fig.savefig(out_dir / "fig4_disengagement_breakpoint.png", dpi=150)
#     plt.close(fig)


# def plot_rolling_slope(df, roll_burnout, roll_di, out_dir):
#     fig, ax1 = plt.subplots(figsize=(11, 5))
#     ax1.plot(df["date"], roll_burnout, color="firebrick", label="rolling slope: burnout_rate")
#     ax1.axhline(0, color="black", linewidth=0.6)
#     ax1.set_ylabel("24-month rolling slope: burnout_rate", color="firebrick")
#     ax1.tick_params(axis="y", labelcolor="firebrick")

#     ax2 = ax1.twinx()
#     ax2.plot(df["date"], roll_di, color="darkorange", label="rolling slope: disengagement index")
#     ax2.set_ylabel("24-month rolling slope: disengagement index", color="darkorange")
#     ax2.tick_params(axis="y", labelcolor="darkorange")

#     _mark_events(ax1)
#     ax1.set_title("Q3: Rolling 24-month trend slope (gradual vs. punctuated)")
#     fig.tight_layout()
#     fig.savefig(out_dir / "fig5_rolling_slope.png", dpi=150)
#     plt.close(fig)


# # ----------------------------------------------------------------------
# # Text report
# # ----------------------------------------------------------------------

# def write_text_report(df, q1, q2, q3, out_dir):
#     lines = []
#     lines.append("=" * 78)
#     lines.append("BURNOUT DISCOURSE TREND ANALYSIS - SUMMARY REPORT")
#     lines.append(f"Data: {df['date'].min():%Y-%m} to {df['date'].max():%Y-%m}  (n={len(df)} months)")
#     lines.append("=" * 78)

#     lines.append("\n--- Q1: Has prevalence increased significantly, 2018-2026? ---")
#     for _, r in q1["table"].iterrows():
#         sig = "SIGNIFICANT" if r["hac_p_value"] < 0.05 else "not significant"
#         lines.append(
#             f"  {r['series']:>16}: HAC-OLS slope={r['hac_slope_per_year']:+.5f}/yr "
#             f"(p={r['hac_p_value']:.4g}, {sig}); Mann-Kendall tau={r['mk_tau']:+.3f} "
#             f"(p={r['mk_p_value']:.4g}); Theil-Sen slope={r['theilsen_slope_per_year']:+.5f}/yr"
#         )
#     lines.append(
#         f"  burnout_rate: {q1['first_year']} mean={q1['first_year_rate']:.4f} -> "
#         f"{q1['last_full_year']} mean={q1['last_full_year_rate']:.4f} "
#         f"({q1['pct_change']:+.1f}% relative change, last full year vs. first year)"
#     )

#     lines.append("\n--- Q2: Do the four constructs grow at different rates? ---")
#     for _, r in q2["construct_trends"].iterrows():
#         lines.append(
#             f"  {r['series']:>22}: HAC-OLS slope={r['hac_slope_per_year']:+.5f}/yr "
#             f"(p={r['hac_p_value']:.4g}); Theil-Sen={r['theilsen_slope_per_year']:+.5f}/yr"
#         )
#     lines.append(
#         f"  Pooled regression, joint Wald test that all four slopes are equal: "
#         f"p={q2['interaction_pvalue']:.4g} "
#         f"({'REJECT equal slopes -> compositional shift' if q2['interaction_pvalue'] < 0.05 else 'fail to reject equal slopes'})"
#     )
#     lines.append("  Share-of-signal trends (is each construct gaining/losing relative weight?):")
#     for _, r in q2["share_trends"].iterrows():
#         sig = "SIGNIFICANT" if r["hac_p_value"] < 0.05 else "n.s."
#         lines.append(
#             f"    {r['series']:>12}: slope={r['hac_slope_per_year']:+.5f}/yr (p={r['hac_p_value']:.4g}, {sig})"
#         )
#     di = q2["disengagement_trend"]
#     lines.append(
#         f"  Disengagement index (EX+MD)/(EMO+COG): slope={di['hac_slope_per_year']:+.5f}/yr "
#         f"(p={di['hac_p_value']:.4g}); Theil-Sen={di['theilsen_slope_per_year']:+.5f}/yr"
#     )

#     lines.append("\n--- Q3: Gradual/secular or punctuated? ---")
#     lines.append("  Acceleration test (significant +quadratic term = accelerating):")
#     for _, r in q3["acceleration"].iterrows():
#         sig = "SIGNIFICANT ACCELERATION" if (r["quadratic_p_value"] < 0.05 and r["quadratic_coef"] > 0) else "no significant acceleration"
#         lines.append(
#             f"    {r['series']:>20}: quadratic coef={r['quadratic_coef']:+.5f}, "
#             f"p={r['quadratic_p_value']:.4g} ({sig})"
#         )
#     lines.append("  Single-breakpoint structural break test (bootstrap p-value, block-bootstrap n=%d):" % N_BOOT)
#     for series_name, row in q3["breakpoints"].iterrows():
#         sig = "SIGNIFICANT BREAK" if row["bootstrap_p_value"] < 0.05 else "not significant"
#         ev_note = (f", within {row['months_from_nearest_event']} mo of {row['nearest_event']}"
#                    if row["within_3_months_of_event"] else
#                    f" ({row['months_from_nearest_event']} mo from nearest named event, {row['nearest_event']})")
#         lines.append(
#             f"    {series_name:>20}: breakpoint={row['breakpoint_date']}, "
#             f"slope before={row['slope_before']:+.5f}/yr, after={row['slope_after']:+.5f}/yr, "
#             f"bootstrap p={row['bootstrap_p_value']:.4g} ({sig}){ev_note}"
#         )
#     lines.append("  Descriptive BIC across 0/1/2 breakpoints (lower = better fit penalized for complexity):")
#     for _, r in q3["bic"].iterrows():
#         lines.append(f"    {r['series']:>20} | k={r['k_breaks']}: BIC={r['bic']:.3f}  breaks={r['breakpoint_dates']}")

#     text = "\n".join(lines)
#     with open(out_dir / "summary_report.txt", "w") as f:
#         f.write(text)
#     return text


# # ----------------------------------------------------------------------
# # Main
# # ----------------------------------------------------------------------

# def main():
#     in_path = "monthly_trend.csv"
#     out_dir = Path(sys.argv[2]) if len(sys.argv) > 2 else Path("./burnout_trend_outputs")
#     out_dir.mkdir(parents=True, exist_ok=True)

#     df = load_data(in_path)

#     q1 = run_q1(df, out_dir)
#     q2 = run_q2(df, out_dir)
#     q3 = run_q3(df, q2["disengagement_series"], out_dir)

#     plot_overall_trend(df, out_dir)
#     plot_indexed_constructs(q2["indexed"], out_dir)
#     plot_shares(q2["shares"], out_dir)
#     plot_disengagement_and_breaks(df, q2["disengagement_series"], q3["breakpoints"], out_dir)
#     plot_rolling_slope(df, q3["rolling_slope_burnout"], q3["rolling_slope_disengagement"], out_dir)

#     report_text = write_text_report(df, q1, q2, q3, out_dir)
#     print(report_text)
#     print(f"\nAll tables, figures, and this report were saved to: {out_dir.resolve()}")


# if __name__ == "__main__":
#     main()

BURNOUT DISCOURSE TREND ANALYSIS - SUMMARY REPORT
Data: 2018-01 to 2026-04  (n=100 months)

--- Q1: Has prevalence increased significantly, 2018-2026? ---
      burnout_rate: HAC-OLS slope=+0.00401/yr (p=9.555e-17, SIGNIFICANT); Mann-Kendall tau=+0.545 (p=8.882e-16); Theil-Sen slope=+0.00390/yr
    bat_score_mean: HAC-OLS slope=+0.01219/yr (p=3.873e-10, SIGNIFICANT); Mann-Kendall tau=+0.497 (p=2.474e-13); Theil-Sen slope=+0.01181/yr
  burnout_rate: 2018 mean=0.0640 -> 2025 mean=0.0881 (+37.7% relative change, last full year vs. first year)

--- Q2: Do the four constructs grow at different rates? ---
              Exhaustion: HAC-OLS slope=+0.00300/yr (p=2.208e-12); Theil-Sen=+0.00300/yr
    Emotional Impairment: HAC-OLS slope=+0.00384/yr (p=5.919e-05); Theil-Sen=+0.00369/yr
    Cognitive Impairment: HAC-OLS slope=+0.00117/yr (p=1.134e-05); Theil-Sen=+0.00120/yr
         Mental Distance: HAC-OLS slope=+0.00418/yr (p=8.844e-12); Theil-Sen=+0.00420/yr
  Pooled regression, joint Wald test 

## ADDing Revised version of analysis plots 
What changed and why
---------------------
fig1 (overall trend):
  - Removed the four CVE-event vertical lines/labels entirely.
  - Added a real calculation instead of decoration: a bottom panel
    showing year-over-year (YoY) percent change in the annual mean
    burnout_rate, with each bar labeled with its exact value. This is
    the concrete "how much did it change, and in which years" answer,
    rather than just eyeballing a dotted trend line.
  - Top panel also marks each year's mean as a black dot and annotates
    the first-year and last-full-year values directly on the plot
    (with the overall % change between them), so the headline number
    is visible without reading a separate table.
  - The partial final year (2026, only Jan-Apr) is excluded from the
    YoY bar chart (it would show a misleading drop just because it's
    a 4-month average compared to a 12-month average), but is kept in
    the saved CSV with a `partial_year` flag for transparency.
 
fig2 (indexed constructs):
  - Switched from 4 overlapping noisy monthly lines on one axes to a
    2x2 small-multiples layout: one panel per construct. This is the
    "sample it for each type" version -- each construct gets its own
    panel instead of competing for the same space.
  - Within each panel, the raw noisy monthly series is shown faint,
    with a 3-month rolling average drawn bold on top, so the
    underlying trend is readable without losing the raw data.
 
This script is self-contained: it only needs monthly_trend.csv and
recomputes the indexed series itself (no dependency on having already
run burnout_trend_analysis.py).
 

In [34]:

import sys
from pathlib import Path
 
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
 
CONSTRUCTS = ["EX", "EMO", "COG", "MD"]
RATE_COLS = {c: f"{c}_yes_rate" for c in CONSTRUCTS}
CONSTRUCT_LABELS = {
    "EX": "Exhaustion",
    "EMO": "Emotional Impairment",
    "COG": "Cognitive Impairment",
    "MD": "Mental Distance",
}
CONSTRUCT_COLORS = {
    "EX": "#1f77b4",
    "EMO": "#ff7f0e",
    "COG": "#2ca02c",
    "MD": "#d62728",
}
 
 
def load_data(path):
    df = pd.read_csv(path)
    df["date"] = pd.to_datetime(df["year_month"], format="%Y-%m")
    df = df.sort_values("date").reset_index(drop=True)
    df["t_month"] = np.arange(len(df))
    df["t_year"] = df["t_month"] / 12.0
    df["year"] = df["date"].dt.year
    return df

In [35]:
#  Figure 1 (revised)
# ----------------------------------------------------------------------
 
def compute_yoy_change(df, value_col="burnout_rate"):
    """Year-over-year change in the annual mean of value_col."""
    yoy = df.groupby("year")[value_col].mean().reset_index()
    yoy["n_months"] = df.groupby("year")["t_month"].count().values
    yoy["partial_year"] = yoy["n_months"] < 12
    yoy["abs_change"] = yoy[value_col].diff()
    yoy["pct_change"] = yoy[value_col].pct_change() * 100
    return yoy
 
 
def plot_fig1_revised(df, out_dir):
    yoy = compute_yoy_change(df, "burnout_rate")
 
    fig, ax1 = plt.subplots(figsize=(11, 5.5))
 
    # --- monthly trend, no event lines ---
    ax1.plot(df["date"], df["burnout_rate"], color="firebrick", linewidth=1.1,
              alpha=0.85, label="monthly burnout_rate")
 
    slope, intercept = np.polyfit(df["t_year"], df["burnout_rate"], 1)
    ax1.plot(df["date"], slope * df["t_year"] + intercept, color="firebrick",
              linestyle=":", linewidth=1.6, label="linear trend")
 
    annual_means = df.groupby("year")["burnout_rate"].mean()
    annual_dates = pd.to_datetime(annual_means.index.astype(str) + "-07-01")
    ax1.scatter(annual_dates, annual_means.values, color="black", zorder=5,
                 s=30, label="annual mean")
 
    # first_year = int(df["year"].min())
    # last_full_year = int(df[df["year"] < df["year"].max()]["year"].max())
    # first_val = annual_means.loc[first_year]
    # last_val = annual_means.loc[last_full_year]
    # pct = 100 * (last_val - first_val) / first_val
 
    # summary_text = (
    #     f"{first_year} mean: {first_val:.1%}  \u2192  {last_full_year} mean: {last_val:.1%}\n"
    #     f"Relative change: {pct:+.1f}%"
    # )
    # ax1.text(
    #     0.99, 0.04, summary_text, transform=ax1.transAxes,
    #     ha="right", va="bottom", fontsize=9.5,
    #     bbox=dict(boxstyle="round,pad=0.4", facecolor="white", alpha=0.9, edgecolor="gray"),
    # )
 
    ax1.set_ylabel("burnout_rate")
    ax1.set_xlabel("Date")
    ax1.set_title("Overall burnout-discourse prevalence, 2018-2026")
    ax1.legend(loc="upper left", fontsize=9)
 
    fig.tight_layout()
    fig.savefig(out_dir / "fig1_overall_trend_v2.png", dpi=150)
    plt.close(fig)
    return yoy

In [36]:
# ----------------------------------------------------------------------
# Figure 2 (revised)
# ----------------------------------------------------------------------
 
def build_indexed(df):
    base_year = df["year"].min()
    baseline = df[df["year"] == base_year][[RATE_COLS[c] for c in CONSTRUCTS]].mean()
    indexed = pd.DataFrame({c: 100 * df[RATE_COLS[c]] / baseline[RATE_COLS[c]] for c in CONSTRUCTS})
    indexed["date"] = df["date"]
    return indexed
 
 
def plot_fig2_revised(df, out_dir, rolling_window=3):
    """Saves one separate PNG per construct, using the raw *_yes_rate columns
    directly from the CSV (no first-year indexing/normalization)."""
    saved_paths = []
 
    for c in CONSTRUCTS:
        color = CONSTRUCT_COLORS[c]
        rate_col = RATE_COLS[c]
        fig, ax = plt.subplots(figsize=(9, 5))
 
        ax.plot(df["date"], df[rate_col], color=color, alpha=0.30, linewidth=1.0,
                 label="monthly")
        rolling = df[rate_col].rolling(rolling_window, center=True).mean()
        ax.plot(df["date"], rolling, color=color, linewidth=2.3,
                 label=f"{rolling_window}-mo avg")
 
        ax.set_title(f"{c} \u2014 {CONSTRUCT_LABELS[c]}: monthly rate", fontsize=12)
        ax.set_xlabel("Date")
        ax.set_ylabel(f"{rate_col} (proportion of posts)")
        ax.legend(fontsize=9, loc="upper left")
 
        fig.tight_layout()
        out_path = out_dir / f"fig2_{c}_rate.png"
        fig.savefig(out_path, dpi=150)
        plt.close(fig)
        saved_paths.append(out_path)
 
    return saved_paths

In [ ]:

# def main():
#     in_path = "monthly_trend.csv"
#     out_dir = Path(sys.argv[2]) if len(sys.argv) > 2 else Path("./burnout_trend_outputs")
#     out_dir.mkdir(parents=True, exist_ok=True)
 
#     df = load_data(in_path)
 
#     yoy = plot_fig1_revised(df, out_dir)
    
#     # build_indexed_plot = plot_fig2_revised(df, out_dir)
 
#     # yoy.to_csv(out_dir / "yoy_change_table.csv", index=False)
 
#     print("Saved: fig1_overall_trend_v2.png")
#     # print()
#     # print(yoy.to_string(index=False))
 
 
# if __name__ == "__main__":
#     main()

Saved: fig1_overall_trend_v2.png


In [37]:

 
def main():
    in_path = "monthly_trend.csv"
    out_dir = Path(sys.argv[2]) if len(sys.argv) > 2 else Path("./burnout_trend_outputs_v2")
    out_dir.mkdir(parents=True, exist_ok=True)
 
    df = load_data(in_path)
 
    yoy = plot_fig1_revised(df, out_dir)
    fig2_paths = plot_fig2_revised(df, out_dir)
 
    yoy.to_csv(out_dir / "yoy_change_table.csv", index=False)
 
    print("Saved: fig1_overall_trend_v2.png")
    for p in fig2_paths:
        print(f"Saved: {p.name}")
    print("Saved: yoy_change_table.csv")
    print()
    print(yoy.to_string(index=False))
 
 
if __name__ == "__main__":
    main()
 

Saved: fig1_overall_trend_v2.png
Saved: fig2_EX_rate.png
Saved: fig2_EMO_rate.png
Saved: fig2_COG_rate.png
Saved: fig2_MD_rate.png
Saved: yoy_change_table.csv

 year  burnout_rate  n_months  partial_year  abs_change  pct_change
 2018      0.064008        12         False         NaN         NaN
 2019      0.059042        12         False   -0.004967   -7.759406
 2020      0.063225        12         False    0.004183    7.085392
 2021      0.075833        12         False    0.012608   19.942006
 2022      0.079717        12         False    0.003883    5.120879
 2023      0.084650        12         False    0.004933    6.188585
 2024      0.084117        12         False   -0.000533   -0.630045
 2025      0.088117        12         False    0.004000    4.755300
 2026      0.081850         4          True   -0.006267   -7.111784
